# Analisis exploratorio de series temporales

El objetivo de este cuaderno es realizar una exploración de las series de tiempo Price_X, Price_Y, Price_Z, Price_Equipo1 y Price_Equipo2. La idea es entender gráfica y analíticamente el comportamiento de ellas, y a su vez, dar paso a un conjunto de pruebas que nos ayudarán a poner a prueba la hipótesis de la gerencia de proyectos en torno relación de dependencia entre el precio de las materia primas y el precio de los equipos. Adicionalmente, estos analisis servirán como base fundacional de los modelos de predicción que se construirán mas adelante.

In [ ]:
import sys
from pathlib import Path
import dataframe_image as dfi

# Importar src.config
BASE_PATH = Path("..").resolve()
if str(BASE_PATH) not in sys.path:
    sys.path.append(str(BASE_PATH))

img_dir = Path("../docs/img")
img_dir.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import plotly.express as px
from src import config
import plotly.graph_objects as go
from plotly.subplots import make_subplots

file_path = config.DATA_RAW_DIR / config.RAW_FILES["historico"]

In [ ]:
df_historico = pd.read_csv(
    file_path,
    parse_dates=["Date"],
    index_col="Date"
)

df_historico = df_historico.sort_index()

#Verificamos la Frecuencia de días hábiles (Business Days)
rango_bdays = pd.date_range(start=df_historico.index.min(), end=df_historico.index.max(), freq='B')
dias_faltantes = rango_bdays.difference(df_historico.index)


print("*" * 70)
print("INSPECCIÓN DE SERIE TEMPORAL: historico_equipos.csv")
print("*" * 70)
print(f"• Dimensiones       : {df_historico.shape[0]} observaciones × {df_historico.shape[1]} variables")
print(f"• Rango de fechas   : {df_historico.index.min().strftime('%Y-%m-%d')} hasta {df_historico.index.max().strftime('%Y-%m-%d')}")
print(f"• Nulos en dataset  : {df_historico.isnull().sum().sum()}")
print(f"• Días hábiles sin datos : {len(dias_faltantes)} (frecuencia esperada: 'B')")
print("=" * 65 + "\n")

print("--- Primeras 5 filas ---")
display(df_historico.head())

print("\n--- Resumen Estadístico (Precios) ---")
display(df_historico.describe().round(2))

INSPECCIÓN DE SERIE TEMPORAL: historico_equipos.csv
• Dimensiones       : 3530 observaciones × 5 variables
• Rango de fechas   : 2010-01-04 hasta 2023-08-31
• Nulos en dataset  : 0
• Días hábiles sin datos : 34 (frecuencia esperada: 'B')

--- Primeras 5 filas ---


,Price_X,Price_Y,Price_Z,Price_Equipo1,Price_Equipo2
Date,,,,,
2010-01-04,80.12,527.5,2225.25,434.73,931.73
2010-01-05,80.59,527.5,2246.50,449.97,968.56
2010-01-06,81.89,527.5,2302.50,444.48,960.51
2010-01-07,81.51,527.5,2306.50,440.90,960.14
2010-01-08,81.37,552.5,2261.25,448.82,949.55



--- Resumen Estadístico (Precios) ---


,Price_X,Price_Y,Price_Z,Price_Equipo1,Price_Equipo2
count,3530.00,3530.00,3530.00,3530.00,3530.00
mean,78.09,555.53,2037.43,460.04,889.98
std,25.19,138.49,373.14,113.68,170.04
min,19.33,257.50,1421.50,208.34,566.00
25%,57.05,482.50,1767.25,398.23,777.70
50%,75.40,541.62,1974.75,451.25,869.78
75%,104.58,620.00,2235.94,515.61,979.12
max,127.98,1062.37,3984.00,855.32,1703.96


Rápidamente lo que nos dice esto es que las series contienen 3530 registros sin datos faltantes. La única excepción son 34 dias que no aparecen registrados por tratarse, posiblemente, de dias festivos. No se especifica una unidad de medición, pero dado que son valores de precio de mercado, podria tratarse de dólares.

In [18]:
series_cols = df_historico.columns.tolist()
colores = [
    "#0052CC",  
    "#00875A",  
    "#FF5630",  
    "#6554C0",  
    "#00B8D9",  
]

titulos_subgraficos = [f"<b>{col}</b>" for col in series_cols] + [
    "<b>Evolución Comparativa (Base 100)</b>"
]

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=titulos_subgraficos,
    horizontal_spacing=0.08,
    vertical_spacing=0.10,
)

for idx, col in enumerate(series_cols):
    row = (idx // 2) + 1
    col_pos = (idx % 2) + 1

    fig.add_trace(
        go.Scatter(
            x=df_historico.index,
            y=df_historico[col],
            mode="lines",
            name=col,
            line=dict(color=colores[idx % len(colores)], width=1.8),
            showlegend=False,  # Ocultamos leyenda individual para evitar desorden
        ),
        row=row,
        col=col_pos,
    )

df_base100 = (df_historico / df_historico.iloc[0]) * 100

for idx, col in enumerate(series_cols):
    fig.add_trace(
        go.Scatter(
            x=df_base100.index,
            y=df_base100[col],
            mode="lines",
            name=col,
            line=dict(color=colores[idx % len(colores)], width=1.5),
            opacity=0.85,
            showlegend=True,  # Mostramos leyenda general aquí
        ),
        row=3,
        col=2,
    )


fig.update_layout(
    title=dict(
        text="<b>Análisis Exploratorio de Series Temporales (Insumos vs. Equipos)</b>",
        font=dict(size=18, color="#172B4D"),
        x=0.02,
    ),
    height=850,
    width=1150,
    template="plotly_white",
    hovermode="x unified",  
    legend=dict(
        title="<b>Series (Base 100)</b>",
        orientation="h",
        yanchor="bottom",
        y=-0.12,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=40, r=40, t=80, b=80),
)

# Mejorar legibilidad de ejes Y en cada gráfico
fig.update_yaxes(showgrid=True, gridcolor="#F4F5F7", zeroline=False)
fig.update_xaxes(showgrid=False)

fig.show()

ruta_imagen = img_dir / "eda_series_base100.png"

fig.write_image(
    ruta_imagen, width=1200, height=800, scale=2
)

print(f"✅ Gráfico guardado en: {ruta_imagen}")

✅ Gráfico guardado en: ..\docs\img\eda_series_base100.png


# Conversión a datos mensuales

Aunque el conjunto de datos original registra observaciones diarias (días hábiles), he decidido realizar una agregación a frecuencia mensual. En la industria de la construcción, los horizontes de planificación, compras de CapEx, cronogramas de proyecto y cláusulas de reajuste de precios contractuales operan sobre ciclos mensuales. Asimismo, los costos de maquinaria pesada presentan rigidez de precios a corto plazo, por lo que el dato diario refleja principalmente ruido transaccional o feriados del mercado. Modular la serie a escala mensual captura con realismo el ciclo contractual de la construcción y mejora sustancialmente la señal predictiva del modelo para el horizonte de toma de decisiones de la gerencia.

Para ser un poco mas rigurosos con la comlpetitud de los datos, lo que hice fue imputar esos 34 valores con el precio del business day anterior para todas las series. Luego de ello, hice un resampling calculando el valor promedio mensual.

In [37]:

rango_bdays = pd.date_range(
    start=df_historico.index.min(), end=df_historico.index.max(), freq="B"
)
df_diario_completo = df_historico.reindex(rango_bdays)

dias_imputados = df_diario_completo.isnull().any(axis=1).sum()
df_diario_completo = df_diario_completo.ffill()

df_mensual = df_diario_completo.resample("ME").mean()

print("*" * 70)
print("📋 INSPECCIÓN DEL DATASET MENSUAL PROCESADO (ffill + resample ME)")
print("*" * 70)
print(f"• Días festivos imputados (ffill)   : {dias_imputados} días hábiles")
print(
    f"• Dimensiones DataFrame mensual     : {df_mensual.shape[0]} meses × {df_mensual.shape[1]} variables"
)
print(
    f"• Rango de fechas                   : {df_mensual.index.min().strftime('%Y-%m')} hasta {df_mensual.index.max().strftime('%Y-%m')}"
)
print(f"• Nulos finales en df_mensual       : {df_mensual.isnull().sum().sum()}")
print("*" * 70 + "\n")

print("--- Primeras 5 observaciones de la serie mensual lista para análisis ---")
display(df_mensual.head())

print("\n--- Resumen Estadístico (Precios) ---")
display(df_mensual.describe().round(2))

**********************************************************************
📋 INSPECCIÓN DEL DATASET MENSUAL PROCESADO (ffill + resample ME)
**********************************************************************
• Días festivos imputados (ffill)   : 34 días hábiles
• Dimensiones DataFrame mensual     : 164 meses × 5 variables
• Rango de fechas                   : 2010-01 hasta 2023-08
• Nulos finales en df_mensual       : 0
**********************************************************************

--- Primeras 5 observaciones de la serie mensual lista para análisis ---


,Price_X,Price_Y,Price_Z,Price_Equipo1,Price_Equipo2
2010-01-31,77.013000,547.000000,2234.325000,451.818000,945.189500
2010-02-28,74.790000,538.000000,2048.575000,444.143500,886.575000
2010-03-31,79.931304,606.630435,2204.130435,501.005217,965.705652
2010-04-30,85.674545,681.250000,2313.829545,560.579545,1025.693182
2010-05-31,76.997619,679.642857,2048.666667,558.783333,932.173333



--- Resumen Estadístico (Precios) ---


,Price_X,Price_Y,Price_Z,Price_Equipo1,Price_Equipo2
count,164.00,164.00,164.00,164.00,164.00
mean,78.07,555.50,2037.55,460.02,889.99
std,25.14,138.08,370.44,113.08,168.73
min,26.85,259.24,1455.80,214.87,588.67
25%,57.23,483.34,1772.19,398.28,781.26
50%,75.51,541.72,1989.62,451.71,871.36
75%,104.30,617.08,2234.37,517.54,970.60
max,124.54,956.73,3537.28,781.56,1499.63


In [36]:
# 1. Asegurar el submuestreo mensual (Media fin de mes)
df_mensual = df_historico.resample("ME").mean()

# 2. Definir paleta profesional y nombres de series
series_cols = df_mensual.columns.tolist()
colores = [
    "#0052CC",  # Azul fuerte
    "#00875A",  # Verde
    "#FF5630",  # Naranja/Rojo
    "#6554C0",  # Morado
    "#00B8D9",  # Cian
]

# 3. Configurar la grilla de subgráficos 3x2
titulos_subgraficos = [f"<b>{col} (Mensual)</b>" for col in series_cols] + [
    "<b>Evolución Comparativa Mensual (Base 100)</b>"
]

fig_mensual = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=titulos_subgraficos,
    horizontal_spacing=0.08,
    vertical_spacing=0.10,
)

# 4. Dibujar los 5 gráficos individuales mensuales (Filas 1 a 3, Columnas 1 y 2)
for idx, col in enumerate(series_cols):
    row = (idx // 2) + 1
    col_pos = (idx % 2) + 1

    fig_mensual.add_trace(
        go.Scatter(
            x=df_mensual.index,
            y=df_mensual[col],
            mode="lines+markers",
            name=col,
            line=dict(color=colores[idx % len(colores)], width=2),
            marker=dict(size=4),  # Marcador discreto para ver el punto mensual
            showlegend=False,
        ),
        row=row,
        col=col_pos,
    )

# 5. Sexto gráfico (Fila 3, Columna 2): Comparativa Normalizada Base 100 Mensual
df_mensual_base100 = (df_mensual / df_mensual.iloc[0]) * 100

for idx, col in enumerate(series_cols):
    fig_mensual.add_trace(
        go.Scatter(
            x=df_mensual_base100.index,
            y=df_mensual_base100[col],
            mode="lines",
            name=col,
            line=dict(color=colores[idx % len(colores)], width=2),
            opacity=0.85,
            showlegend=True,
        ),
        row=3,
        col=2,
    )

# 6. Formatear diseño global del dashboard
fig_mensual.update_layout(
    title=dict(
        text="<b>Análisis Exploratorio de Series Temporales - Frecuencia Mensual (Insumos vs. Equipos)</b>",
        font=dict(size=18, color="#172B4D"),
        x=0.02,
    ),
    height=850,
    width=1150,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        title="<b>Series (Base 100)</b>",
        orientation="h",
        yanchor="bottom",
        y=-0.12,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=40, r=40, t=80, b=80),
)

fig_mensual.update_yaxes(showgrid=True, gridcolor="#F4F5F7", zeroline=False)
fig_mensual.update_xaxes(showgrid=False)

fig_mensual.show()

ruta_imagen_mes = img_dir / "eda_series_base100_mes.png"

fig_mensual.write_image(
    ruta_imagen_mes, width=1200, height=800, scale=2
)

print(f"✅ Gráfico guardado en: {ruta_imagen_mes}")

✅ Gráfico guardado en: ..\docs\img\eda_series_base100_mes.png


**Nota 1:** Al menos en la inspección visual, las series *Price_Z* y *Price_Equipo2* y *Price_Y* y *Price_Equipo1* parecen ser idénticas entre si (o al menos parecen estar relacionadas por un factor multiplicativo en la escala). Este efecto visual podria indicar que estan correlacionadas. Sin embargo, no seria correcto decir que una explica a la otra ya que en el fondo podria tratarse de correlación espuria. Un tratamiento correcto para determinar tal relación es realizando pruebas de estacionariedad, causalidad (como la de Granger) u observando las series diferenciadas. 

# Pruebas de estacionariedad y causalidad

En esta sección voy a realizar algunas pruebas iniciales que permitirán dilucidar las posibles relaciones entre las variables.
En series financieras y de precios (como es este ejercicio), lo normal es que tengan raíz unitaria $I(1)$. Esto simplemente significa si la serie es volatil o no, es decir, si su media y varianza y autocorrelación se mantienen constantes en el tiempo o no.
Primero probaré en niveles (empezar con la serie cruda) y, si no se rechaza $H_0$ (serie no estacionaria), aplicaré primera diferencia logarítmica (tasas de crecimiento / retornos), que suele estabilizar tanto la media como la varianza. El test usual en este caso es el Test de Dickey-Fuller.

## Test de Estacionariedad

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR

# ==============================================================================
# PASO 1: VERIFICACIÓN DE ESTACIONARIEDAD (TEST ADF)
# H0: La serie tiene raíz unitaria (No Estacionaria)
# H1: La serie es estacionaria
# ==============================================================================


def test_adf_resumen(df, variables, alpha=0.01):
    resultados = []
    for var in variables:
        # Test en niveles
        adf_lvl = adfuller(df[var].dropna(), autolag="AIC")
        p_val_lvl = adf_lvl[1]

        # Test en primeras diferencias logarítmicas (tasas de crecimiento)
        diff_log = np.log(df[var] / df[var].shift(1)).dropna()
        adf_diff = adfuller(diff_log, autolag="AIC")
        p_val_diff = adf_diff[1]

        resultados.append(
            {
                "Variable": var,
                "p-val (Niveles)": round(p_val_lvl, 4),
                "Estacionaria en Niveles?": "Sí I(0)"
                if p_val_lvl < alpha
                else "No I(1)",
                "p-val (Diff Log)": round(p_val_diff, 4),
                "Estacionaria en Diff Log?": "Sí I(0)"
                if p_val_diff < alpha
                else "No I(1)",
            }
        )
    return pd.DataFrame(resultados)


cols_evaluar = [
    "Price_X",
    "Price_Y",
    "Price_Z",
    "Price_Equipo1",
    "Price_Equipo2",
]
df_adf = test_adf_resumen(df_mensual, cols_evaluar)

print("=" * 90)
print("📊 RESULTADOS DEL TEST DE ESTACIONARIEDAD (ADF - H0: Raíz Unitaria) - alpha = 0.01")
print("=" * 90)
display(df_adf)

dfi.export(df_adf, img_dir / 'df_adf.pdf')

📊 RESULTADOS DEL TEST DE ESTACIONARIEDAD (ADF - H0: Raíz Unitaria) - alpha = 0.01


,Variable,p-val (Niveles),Estacionaria en Niveles?,p-val (Diff Log),Estacionaria en Diff Log?
0,Price_X,0.4046,No I(1),0.0000,Sí I(0)
1,Price_Y,0.3004,No I(1),0.0009,Sí I(0)
2,Price_Z,0.0425,No I(1),0.0000,Sí I(0)
3,Price_Equipo1,0.3123,No I(1),0.0008,Sí I(0)
4,Price_Equipo2,0.0290,No I(1),0.0000,Sí I(0)


**Nota 2:** Lo que esta tabla nos dice es que las series no son estacionarias de forma pura ya que  los p-values de prueba son significativamente mayores a 0.01. Se ha elegido este valor de alfa tomando en cuenta la baja potencia del test ADF en muestras finitas ($N \approx 150$) (Enders, 2014; Schwert, 1989). De acuerdo con los autores, el test presenta una alta tasa de falsos positivos al diferenciar entre un verdadero proceso con raíz unitaria y uno estacionario con alta persistencia. No podemos rechazar $H_0$; por lo tanto, no son estacionarias en niveles (tienen tendencia o inflación acumulada). Bajo el umbral formal del 1% ($\alpha = 0.01$), no se rechaza $H_0$ en niveles para ninguna de las cinco series, mientras que todas rechazan contundentemente $H_0$ en primera diferencia logarítmica ($p \le 0.0009$). Esto finalmente nos dice que las cinco variables se clasifican teórica y empíricamente como procesos integrados de orden uno ($I(1)$), es decir: podria tratarse de una caminata aleatoria.


## Obtención de rezagos óptimos

Para no adivinar el número de retardos (lags) en el test de Granger, ajustamos un modelo VAR preliminar con las series estacionarias y dejamos que los Criterios de Información (AIC, BIC, HQIC) nos sugieran la memoria temporal óptima del sistema. Tambien calculamos una matriz de Granger que nos permita conocer la causalidad en los pares de series predictora y objetivo.

In [63]:
def matriz_granger(df_stat, predictores, objetivos, max_lags=12):
    """
    Evalúa la causalidad de Granger y extrae el p-valor de la prueba F (SSR based F-test)
    para cada combinación (Predictor -> Objetivo).
    """
    resultados_granger = []

    for obj in objetivos:
        for pred in predictores:
            # En statsmodels, la serie objetivo va en la columna 0 y el predictor en la 1
            data_test = df_stat[[obj, pred]]
            test_out = grangercausalitytests(
                data_test, maxlag=max_lags,
            )

            # Extraer p-valor de la prueba F para el rezago específico
            p_vals = {
                f"Lag {lag}": round(
                    test_out[lag][0]["ssr_ftest"][1], 4
                )  # índice 1 es el p-val
                for lag in range(1, max_lags + 1)
            }

            row = {"Objetivo": obj, "Predictor (Causa?)": pred}
            row.update(p_vals)
            resultados_granger.append(row)

    return pd.DataFrame(resultados_granger)


In [65]:
# ==============================================================================
# PASO 3: SELECCIÓN DE LAGS VAR Y GRANGER CAUSALITY MENSUAL
# ==============================================================================

# Transformar a retornos logarítmicos mensuales estacionarios I(0)
df_mensual_stat = np.log(df_mensual / df_mensual.shift(1)).dropna()
max_lags = 12  # Evaluar hasta 12 meses de memoria

modelo_var_m = VAR(df_mensual_stat)
criterios_var_m = modelo_var_m.select_order(maxlags=max_lags)

print("*" * 70)
print("⏱️ CRITERIOS DE INFORMACIÓN VAR (MEMORIA EN MESES)")
print("*" * 70)
print(criterios_var_m.summary())

lag_opt_mensual = criterios_var_m.aic
print(f"\n👉 Lag óptimo sugerido por AIC en meses: p = {lag_opt_mensual}")


# Calculamos la matriz de Granger

df_granger_mensual = matriz_granger(
    df_mensual_stat,
    predictores=["Price_X", "Price_Y", "Price_Z"],
    objetivos=["Price_Equipo1", "Price_Equipo2"],
    max_lags=max_lags,
)

print("\n" + "=" * 85)
print(
    "🎯 P-VALORES DE CAUSALIDAD DE GRANGER MENSUAL (H0: Insumo NO causa a Equipo)"
)
print(
    "   * Un p < 0.05 en 'Lag 3', por ejemplo, prueba que el insumo anticipa el precio 3 meses *"
)
print("=" * 85)
display(df_granger_mensual)

**********************************************************************
⏱️ CRITERIOS DE INFORMACIÓN VAR (MEMORIA EN MESES)
**********************************************************************
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -38.32      -38.22   2.276e-17      -38.28
1       -39.00     -38.40*   1.155e-17     -38.76*
2      -39.07*      -37.97  1.075e-17*      -38.63
3       -38.98      -37.39   1.178e-17      -38.33
4       -38.86      -36.76   1.343e-17      -38.01
5       -38.78      -36.18   1.471e-17      -37.72
6       -38.73      -35.64   1.553e-17      -37.47
7       -38.57      -34.97   1.864e-17      -37.11
8       -38.51      -34.41   2.019e-17      -36.85
9       -38.40      -33.81   2.317e-17      -36.54
10      -38.30      -33.21   2.668e-17      -36.23
11      -38.30      -32.70   2.811e-17      -36.02
12      -38.11      -32.01   3.608e-17    

,Objetivo,Predictor (Causa?),Lag 1,Lag 2,Lag 3,Lag 4,Lag 5,Lag 6,Lag 7,Lag 8,Lag 9,Lag 10,Lag 11,Lag 12
0,Price_Equipo1,Price_X,0.0499,0.0093,0.0141,0.0333,0.0689,0.0964,0.1166,0.0572,0.0343,0.0643,0.0480,0.0394
1,Price_Equipo1,Price_Y,0.7715,0.7478,0.8388,0.8915,0.9524,0.9782,0.9899,0.9762,0.9774,0.9734,0.9808,0.9860
2,Price_Equipo1,Price_Z,0.0007,0.0004,0.0016,0.0013,0.0058,0.0045,0.0090,0.0041,0.0086,0.0236,0.0376,0.0610
3,Price_Equipo2,Price_X,0.2179,0.4085,0.3269,0.3427,0.3426,0.3855,0.4954,0.4867,0.3350,0.3720,0.3768,0.3963
4,Price_Equipo2,Price_Y,0.4446,0.1945,0.2997,0.4776,0.6927,0.6038,0.4343,0.2283,0.2409,0.1565,0.2079,0.2211
5,Price_Equipo2,Price_Z,0.6417,0.7759,0.8172,0.9267,0.8481,0.8373,0.3624,0.3093,0.1829,0.2569,0.2458,0.2509


El valor del lag óptimo: p = 2 meses lo que indica es el rezago que según la prueba de vectores autoregresivos es el adecuado para predecir el valor futuro de la serie. El Criterio BIC penaliza la inclusión de variables adicionales de forma más severa en muestras pequeñas, sugiriendo un rezago óptimo de $p = 1$. Sin embargo, se priorizó el Criterio AIC ($p = 2$, coincidente con el criterio FPE), ya que en problemas de pronóstico predictivo es preferible preservar la dinámica temporal completa de los 60 días de traspaso de costos, evitando el sesgo por omisión de variables que generaría cortar la memoria en un solo mes.

Mas abajo, la matriz de causalidad de Granger, muestra los resultados de las pruebas de hipótesis nula de Granger para cada par predictor-objetivo en diferentes rezagos:

- $H_0$: La serie del Insumo ($X, Y$ o $Z$) NO causa en el sentido de Granger al Precio del Equipo. \
- $H_1$: La historia del Insumo mejora estadísticamente la predicción del Equipo.

Se puede apreciar, por ejemplo que para el equipo 1, tanto las series X y Z son aparentes aportantes de causalidad de Granger hacia en el precio del Equipo 1 en los horizontes de varios meses (p-value < 0.05). Sin embargo, para el caso del equipo 2, el test concluye que ningún insumo causa en el sentido de Granger al Equipo 2. Lo que esto significa es que la mecánica de fijación de precios del Equipo 2 es instantánea y el test de Granger es completamente inutil para capturar esta estructura.  Cuando el insumo Z o el insumo Y suben de precio en un mes calendario, el fabricante del Equipo 2 ajusta su lista de precios en ese exacto mismo mes ($t = 0$). Esto es justamente lo que mencionaba Jiménez-Rodríguez (2022) respecto al efecto **pass-through**.

En el fondo esto nos dice que al parecer el Equipo 1 tiene que ver con fabricantes con contratos a precio fijo temporal o inventarios de amortiguación, ya que los precios de X y Z son predictivos y su ajuste de precio se va trasladando paulatinamente a lo largo de los meses.

Por el contrario, el equipo 2 tiene mas que ver con fabricantes que venden con cláusulas de indexación inmediata al precio del día o compras contra pedido directo. Basicamente, al cambiar el precio de los insumos, inmediatamente cambia el precio del equipo.

# Test de cointegración de Engle-Granger

El test de cointegración es una prueba que se realiza para encontrar una tendencia estocástica entre dos series no estacionarias que se mueven en la misma dirección. Este test es importante ya que vimos que nuestras series no son estacionarias y además que, al menos visualmente, parece existir una especie de relación entre pares de ellas (ejemplo Price_Z y Price_Equipo2).

In [67]:
from statsmodels.tsa.stattools import coint
import scipy.stats as stats


print("=" * 80)
print("🔗 1. TEST DE COINTEGRACIÓN DE ENGLE-GRANGER (SERIES EN NIVELES I(1))")
print("   H0: Las series NO están cointegradas (no hay equilibrio de largo plazo)")
print("   H1: Las series SÍ están cointegradas (mueven juntas a largo plazo)")
print("=" * 80)

resultados_coint = []
insumos = ["Price_X", "Price_Y", "Price_Z"]
equipos = ["Price_Equipo1", "Price_Equipo2"]

for eq in equipos:
    for ins in insumos:
        # coint devuelve: (t-stat, p-value, critical_values)
        t_stat, p_val, crit = coint(
            df_mensual[eq], df_mensual[ins], trend="c", autolag="AIC"
        )
        resultados_coint.append(
            {
                "Equipo (Objetivo)": eq,
                "Insumo": ins,
                "p-valor Cointegración": round(p_val, 4),
                "¿Cointegradas (p < 0.05)?": "✅ SÍ (Equilibrio L.P.)"
                if p_val < 0.05
                else "❌ NO",
            }
        )

df_coint_res = pd.DataFrame(resultados_coint)
display(df_coint_res)

print("\n" + "=" * 80)
print(
    "⚡ 2. CORRELACIÓN CONTEMPORÁNEA EN NIVELES Y RETORNOS (LAG 0 - SIMULTÁNEA)"
)
print("=" * 80)

# ==============================================================================
# FUNCIÓN DE CORRELACIÓN CRUZADA (CCF): NIVELES Y RETORNOS MENSUALES (LAGS 0 A 6)
# ==============================================================================

# 1. Calcular retornos logarítmicos mensuales I(0)
df_retornos = np.log(df_mensual / df_mensual.shift(1)).dropna()

insumos = ["Price_X", "Price_Y", "Price_Z"]
equipos = ["Price_Equipo1", "Price_Equipo2"]
max_lags = 6

resultados_ccf = []

for eq in equipos:
    for ins in insumos:
        # 2. Calcular la correlación contemporánea en NIVELES I(1)
        data_niveles = df_mensual[[eq, ins]].dropna()
        r_niveles, _ = stats.pearsonr(data_niveles[eq], data_niveles[ins])

        # 3. Inicializar la fila ubicando "Corr. en Niveles (r)" justo después del insumo
        fila = {
            "Equipo (Objetivo)": eq,
            "Insumo (Predictor)": ins,
            "Corr. en Niveles (r)": round(r_niveles, 4),
        }

        # 4. Iterar desde Lag 0 (Contemporáneo) hasta Lag 6 en RETORNOS
        for lag in range(max_lags + 1):
            if lag == 0:
                s_eq = df_retornos[eq]
                s_ins = df_retornos[ins]
            else:
                # Alineamos el Equipo en (t) con el Insumo en (t - lag)
                s_eq = df_retornos[eq].iloc[lag:]
                s_ins = df_retornos[ins].shift(lag).dropna()

            r_val, _ = stats.pearsonr(s_eq, s_ins)
            fila[f"Lag {lag} (r)"] = round(r_val, 4)

        resultados_ccf.append(fila)

df_ccf_res = pd.DataFrame(resultados_ccf)

print("=" * 105)
print(
    "📊 MATRIZ DE CORRELACIÓN CRUZADA (Niveles Contemporáneos + Retornos Lags 0 a 6)"
)
print(
    "   * Mide la fuerza lineal en niveles y en Δln(Equipo_t) vs Δln(Insumo_{t-k}) en escala [-1, 1] *"
)
print("=" * 105)
display(df_ccf_res)

🔗 1. TEST DE COINTEGRACIÓN DE ENGLE-GRANGER (SERIES EN NIVELES I(1))
   H0: Las series NO están cointegradas (no hay equilibrio de largo plazo)
   H1: Las series SÍ están cointegradas (mueven juntas a largo plazo)


,Equipo (Objetivo),Insumo,p-valor Cointegración,¿Cointegradas (p < 0.05)?
0,Price_Equipo1,Price_X,0.5542,❌ NO
1,Price_Equipo1,Price_Y,0.2029,❌ NO
2,Price_Equipo1,Price_Z,0.0342,✅ SÍ (Equilibrio L.P.)
3,Price_Equipo2,Price_X,0.5650,❌ NO
4,Price_Equipo2,Price_Y,0.0144,✅ SÍ (Equilibrio L.P.)
5,Price_Equipo2,Price_Z,0.0268,✅ SÍ (Equilibrio L.P.)



⚡ 2. CORRELACIÓN CONTEMPORÁNEA EN NIVELES Y RETORNOS (LAG 0 - SIMULTÁNEA)
📊 MATRIZ DE CORRELACIÓN CRUZADA (Niveles Contemporáneos + Retornos Lags 0 a 6)
   * Mide la fuerza lineal en niveles y en Δln(Equipo_t) vs Δln(Insumo_{t-k}) en escala [-1, 1] *


,Equipo (Objetivo),Insumo (Predictor),Corr. en Niveles (r),Lag 0 (r),Lag 1 (r),Lag 2 (r),Lag 3 (r),Lag 4 (r),Lag 5 (r),Lag 6 (r)
0,Price_Equipo1,Price_X,0.5292,0.3374,0.2766,0.0630,-0.1261,-0.1323,0.0217,0.0637
1,Price_Equipo1,Price_Y,0.9991,0.9958,0.4257,-0.0463,-0.0633,0.0091,-0.0278,-0.0348
2,Price_Equipo1,Price_Z,0.8550,0.4417,0.4035,0.1458,-0.0494,0.0521,0.0685,-0.0760
3,Price_Equipo2,Price_X,0.5350,0.5061,0.2523,0.0452,-0.1118,-0.1321,0.0319,0.0197
4,Price_Equipo2,Price_Y,0.9217,0.6519,0.2659,-0.0302,-0.0409,0.0389,0.1170,0.1044
5,Price_Equipo2,Price_Z,0.9874,0.9570,0.3169,0.0331,-0.0120,0.0071,0.0915,0.0094


Lo que nos dicen estas tablas es lo siguiente:

- Price_Z está cointegrado tanto con el Equipo 1 ($p = 0.0342$) como con el Equipo 2 ($p = 0.0268$). Esto podria indicar que el insumo Z dicta el precio de equilibrio a largo plazo de ambos equipos.
- El test de Granger decia que nada explicaba al Equipo 2. Y la razón es la siguiente: Tiene un equilibrio de largo plazo con Price_Z ($p = 0.0268$) y con Price_Y ($p = 0.0144$). Su correlación en retornos en el mismo mes (Lag 0) con Price_Z es de 0.9570 (95.5%) y en niveles de 0.9874. Si miramos la correlación en el lag 1(r), baja a 0.31 y luego en lag 2(r) a 0.0331 Esto demuestra que el fabricante del Equipo 2 ajusta sus precios de forma instantánea e indexada a los insumos Z e Y en el mismo mes.

# Conclusiones

- Se determinó que todas las series son procesos integrados de orden uno ($I(1)$ en niveles al umbral estricto de $\alpha = 0.01$) y estrictamente estacionarios en primera diferencia logarítmica ($I(0)$ con $p \le 0.0009$). Este hallazgo habilita el uso de retornos mensuales sin riesgo de regresiones espurias (se equilibran a largo plazo).
- El test de Engle-Granger confirmó la existencia de cointegración estadística entre los precios de los equipos y la materia prima Price_Z ($p = 0.0342$ para Equipo 1 y $p = 0.0268$ para Equipo 2). Esto demuestra que, más allá de los choques mes a mes, el insumo Z define la trayectoria de equilibrio de largo plazo de la industria.
- El cruce entre los tests de Causalidad de Granger, Criterios de Información (AIC/BIC) y Correlación Cruzada (CCF) reveló que los dos equipos operan bajo modelos comerciales completamente opuestos.
- El Precio_Equipo 1 absorbe los costos de las materias primas con un retraso de 1 a 3 meses (confirmado por un orden óptimo AIC $p=2$ en el sistema VAR y significancia en Granger hasta 6 meses). Sus variables predictoras clave son Price_Z y Price_X. El insumo Price_Y queda descartado al no presentar evidencia de causalidad ni correlación útil.
El precio  del Equipo 2 reacciona en el exacto mismo mes en el que cambian los insumos ($Lag\ 0$). Por esta razón, el test de Granger tradicional no detecta causalidad en rezagos pasados ($p > 0.05$). Comparte una altísima correlación contemporánea en niveles ($r = 0.9874$) y en retornos ($r = 0.9552$) con Price_Z, complementada por Price_Y ($r = 0.9217$ en niveles).

In [71]:
import os

# '../' retrocede de /notebooks a la raíz del proyecto
ruta_carpeta = "../data/processed"
os.makedirs(ruta_carpeta, exist_ok=True)

# Guardar el archivo en la ruta correcta
df_mensual.to_parquet(f"{ruta_carpeta}/df_mensual.parquet")

print("✅ Guardado en: prediccion-costos-construccion/data/processed/df_mensual.parquet")

✅ Guardado en: prediccion-costos-construccion/data/processed/df_mensual.parquet
